# 9. Caso de Estudio: West Virginia

West Virginia — el estado con la tasa de mortalidad ajustada por edad más alta del país en 2017 (957.5 por 100,000 habitantes), casi el doble que Hawái con 584.9, el estado con la tasa más baja. Este caso examina si las disparidades geográficas son persistentes o responden a fluctuaciones puntuales.

## 9.1. Evolución de las causas de muerte en West Virginia (1999–2017)

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv("NCHS_Leading_Causes.csv", dtype=str)
df.columns = ['year','cause_113','cause_name','state','deaths','age_adjusted_death_rate']
df['year'] = pd.to_numeric(df['year'], errors='coerce')
df['deaths'] = pd.to_numeric(df['deaths'].str.replace('.','',regex=False).str.replace(',','',regex=False), errors='coerce').fillna(0).astype(int)
df['age_adjusted_death_rate'] = pd.to_numeric(df['age_adjusted_death_rate'].str.replace(',','.'), errors='coerce')
df['state'] = df['state'].str.strip()
df['cause_name'] = df['cause_name'].str.strip()
df = df[(df['year']>=1999)&(df['year']<=2017)].dropna(subset=['year','age_adjusted_death_rate'])

estados = df[df['state']!='United States']
us = df[df['state']=='United States']
estados_2017 = estados[estados['year']==2017]
sin_all = estados[estados['cause_name']!='All causes']
print(f"Datos cargados: {len(df):,} filas | {df['state'].nunique()} entidades | {df['year'].min():.0f}–{df['year'].max():.0f}")


Datos cargados: 10,840 filas | 52 entidades | 1999–2017


In [2]:
wv = estados[(estados['state']=='West Virginia')&(estados['cause_name']!='All causes')].copy()

fig = go.Figure()
causas_wv = wv['cause_name'].unique()
for causa in causas_wv:
    sub = wv[wv['cause_name']==causa].sort_values('year')
    if causa == 'Unintentional injuries':
        fig.add_trace(go.Scatter(x=sub['year'], y=sub['age_adjusted_death_rate'],
                                 mode='lines+markers', name=causa,
                                 line=dict(color='#D62828', width=2.5),
                                 marker=dict(size=7)))
    else:
        fig.add_trace(go.Scatter(x=sub['year'], y=sub['age_adjusted_death_rate'],
                                 mode='lines', name=causa,
                                 line=dict(color='#9AA0A6', width=1),
                                 opacity=0.6, showlegend=False))

fig.add_vline(x=2014, line_dash='dash', line_color='#D62828', opacity=0.5,
              annotation_text='2014: punto de aceleración',
              annotation_font_color='#D62828')
fig.update_layout(
    title='Evolución de las causas de muerte en West Virginia (1999–2017)',
    xaxis_title='Año', yaxis_title='Tasa ajustada por 100,000 hab.',
    height=480, template='plotly_white',
    xaxis=dict(tickmode='linear', tick0=1999, dtick=2),
    legend=dict(orientation='h', y=-0.15)
)
fig.show()

### 9.1.1. ¿Qué explica el crecimiento de las lesiones no intencionales en West Virginia?

No es casual que la diferencia de West Virginia contra la media nacional sea en la categoría de lesiones no intencionadas. Se identifican 5 categorías documentadas por el Programa de Prevención de Violencia y Lesiones de West Virginia (WVDHHR):

1. **Sobredosis con medicinas recetadas:** A partir de los años noventa, médicos comenzaron a recetar en masa opiáceos. En 2012 se entregaron 259 millones de recetas de analgésico en EE.UU. WV absorbió ese impacto de lleno. Las hospitalizaciones relacionadas con drogas aumentaron de 363.7 a 506.5 por cada 10,000 egresos entre 2007 y 2011.

2. **Traumatismo craneoencefálico (TBI):** En 2015 fallecieron 411 residentes — el 20% de todas las defunciones por lesión. El 40.9% de las muertes anuales por TBI corresponden a mayores de 65 años, principalmente por caídas.

3. **Accidentes de circulación:** En 2015, 341 personas murieron en choques vehiculares (17% de todas las muertes por lesión). Primera causa de muerte para personas entre 5 y 24 años.

4. **Abuso infantil y negligencia:** En 2015, cinco niños de 0 a 5 años murieron debido al maltrato infantil.

5. **Violencia de pareja íntima:** Factor de riesgo documentado en el perfil estatal de lesiones.

Estos cinco factores no son problemas aislados. Son diferentes expresiones de una misma **vulnerabilidad estructural**: estado con depresión económica, población envejecida, áreas rurales con acceso limitado a sanidad y una historia de dependencias hacia industrias en desaparición.

## 9.2. Comparación de muertes por causa: West Virginia 1999 vs. 2017

In [3]:
wv_comp = wv[wv['year'].isin([1999,2017])].copy()
orden = wv_comp[wv_comp['year']==2017].sort_values('deaths')['cause_name'].tolist()

fig = go.Figure()
colors_yr = {1999:'#457B9D', 2017:'#1D3557'}
for yr in [1999, 2017]:
    sub = wv_comp[wv_comp['year']==yr].set_index('cause_name').reindex(orden).reset_index()
    fig.add_trace(go.Bar(
        y=sub['cause_name'], x=sub['deaths'],
        name=str(yr), orientation='h',
        marker_color=colors_yr[yr],
        text=sub['deaths'].apply(lambda x: f"{x:,}"),
        textposition='outside'
    ))
fig.update_layout(
    title='West Virginia – Muertes por causa: 1999 vs. 2017',
    xaxis_title='Número de muertes', barmode='group',
    height=500, template='plotly_white',
    legend=dict(orientation='h', y=1.05)
)
fig.show()

## 9.3. West Virginia vs. Promedio Nacional (1999–2017)

In [4]:
wv_total = (estados[(estados['state']=='West Virginia')&(estados['cause_name']=='All causes')]
              .sort_values('year')[['year','age_adjusted_death_rate']])
nac_prom = (estados[estados['cause_name']=='All causes']
              .groupby('year')['age_adjusted_death_rate'].mean().reset_index())

brecha = wv_total.merge(nac_prom, on='year', suffixes=('_wv','_nac'))
brecha['brecha'] = brecha['age_adjusted_death_rate_wv'] - brecha['age_adjusted_death_rate_nac']
brecha_prom = brecha['brecha'].mean()

# Usar .iloc para evitar IndexError
brecha_ini = brecha.iloc[0]['brecha'] if len(brecha) > 0 else 0
brecha_fin = brecha.iloc[-1]['brecha'] if len(brecha) > 0 else 0

print(f"Brecha inicio: {brecha_ini:.1f} | Brecha fin: {brecha_fin:.1f} | Promedio: {brecha_prom:.1f}")

fig = go.Figure()
fig.add_traces([
    go.Scatter(x=list(brecha['year'])+list(brecha['year'][::-1]),
               y=list(brecha['age_adjusted_death_rate_wv'])+list(brecha['age_adjusted_death_rate_nac'][::-1]),
               fill='toself', fillcolor='rgba(230,57,70,0.12)',
               line=dict(color='rgba(0,0,0,0)'), showlegend=False),
    go.Scatter(x=wv_total['year'], y=wv_total['age_adjusted_death_rate'],
               mode='lines+markers', name='West Virginia',
               line=dict(color='#E63946', width=2.2)),
    go.Scatter(x=nac_prom['year'], y=nac_prom['age_adjusted_death_rate'],
               mode='lines+markers', name='Promedio Nacional',
               line=dict(color='#1D3557', width=2.2, dash='dash'))
])
fig.add_annotation(x=brecha['year'].mean(), y=brecha['age_adjusted_death_rate_wv'].mean()-100,
                   text=f"Brecha promedio:<br>+{brecha_prom:.1f} por 100,000 hab.",
                   font=dict(color='#E63946', size=11), showarrow=False)
fig.update_layout(title='West Virginia vs. Promedio Nacional – Todas las causas',
                  xaxis_title='Año', yaxis_title='Tasa por 100,000 hab.',
                  height=460, template='plotly_white',
                  xaxis=dict(tickmode='linear', tick0=int(brecha['year'].min()), dtick=2),
                  legend=dict(orientation='h', y=-0.15))
fig.show()

# Tabla de brecha
tabla_brecha = brecha[brecha['year'].isin(brecha['year'].unique()[::len(brecha)//6])].copy()
tabla_brecha.columns = ['Año','WV (tasa)','Nacional (promedio)','Brecha (WV − Nac.)']
print("\nBrecha entre West Virginia y el promedio nacional:")
print(tabla_brecha.round(1).to_string(index=False))

Brecha inicio: 151.3 | Brecha fin: 201.4 | Promedio: 173.4



Brecha entre West Virginia y el promedio nacional:
 Año  WV (tasa)  Nacional (promedio)  Brecha (WV − Nac.)
2002      999.0                847.7               151.3
2005      965.1                815.1               150.0
2007      954.2                789.6               164.6
2009      947.7                765.1               182.6
2011      953.2                760.2               193.0
2013      923.8                751.2               172.6
2015      943.4                757.1               186.3
2017      957.1                755.7               201.4


**Interpretación:** El hallazgo central es que la brecha absoluta entre West Virginia y el promedio nacional no se cierra. West Virginia mejora, pero no converge. El estado avanza en paralelo al país sin acortar la distancia que lo separa de la media nacional, confirmando la hipótesis sobre la persistencia estructural de las disparidades documentada en la literatura.